# Ирисы Фишера - обучение логистической регрессии

In [ ]:
# Установка зависимостей
!pip install -q scikit-learn numpy pandas seaborn phik

In [ ]:
# Инициализация библиотек
import matplotlib as mpl
mpl.rcParams['figure.constrained_layout.use'] = True

import pandas as pd
import phik

In [ ]:
# Импорт датасета
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

iris.keys()

In [ ]:
print('Описание датасета:')
print(iris.DESCR)

In [ ]:
# Соберем данные вместе с целевым признаком для анализа
iris_df = iris.data.join(iris.target)

# Посмотрим на первые несколько строк
iris_df.head()

**Промежуточный вывод:** датасет Iris небольшой (всего 150 наблюдений), но для начала изучения и первых экспериментов с машинным обучением он отлично подходит.

Также в данных нет выбросов, дубликатов и пропущенных значений - это упрощает процесс EDA

Посмотрим на распределение признаков по классам

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

features_len = len(iris.feature_names)

fig, axes = plt.subplots(
    1,
    features_len,
    figsize=(features_len * 4, 5)
)

fig.suptitle('Сравнение признаков по классам')

for i, feature in enumerate(iris.feature_names):
    sns.boxplot(
        x=iris_df['target'].map(dict(enumerate(iris.target_names))),
        y=iris_df[feature],
        ax=axes[i]
    )

    axes[i].set_title(feature)

fig.set_constrained_layout_pads( wspace=0.1)
plt.show()

Уже по графикам выше видно, что классы хорошо разделяются. Ящики почти не пересекаются, сильнее пересекаются усы и единичные значения выходящие за межквартильный размах.

Построим еще графики корреляций. Для всех видов в общем и по каждому виду отдельно

In [ ]:
# Общий график корреляции
sns.heatmap(
    iris_df.phik_matrix(interval_cols=iris_df.drop(columns=['target']).columns),
    annot=True,
    cmap='coolwarm'
)
plt.show()

Phik показывает корреляцию даже выше чем заявлено в описании датасета. Данные на полученной тепловой карте хорошо коррелируют с данными полученными из ящиков с усами. Чем меньше пересчение ящиков и их усов, тем выше корреляция. Поэтому для обучения моделей может быть вполне достаточно двух признаков `petal length (cm)` и `petal width (cm)`

Посмотрим, дадут ли дополнительную информацию отдельные тепловые карты по каждому таргету

In [ ]:
# Отдельная тепловая карта на каждый таргет

fig, axes = plt.subplots(
    1,
    3,
    figsize=(features_len * 4, 5)
)

fig.suptitle('Корреляция признаков по классам')

for i, feature in enumerate(iris.target_names):
    sns.heatmap(
        iris_df[iris_df['target'] == i].drop(columns=['target']).phik_matrix(interval_cols=iris_df.columns),
        annot=True,
        cmap='coolwarm',
        ax=axes[i],
        fmt=".2f"
    )

    axes[i].set_title(feature)

plt.show()

Для каждого класса тепловая карта сильно отличается. Можно сделать вывод, что в данных достаточно информации для классификации.

Посмотрим дополнительно на то как площадь petal и sepal коррелирует с target

In [ ]:
iris_df_areas = pd.DataFrame({
    'sepal_area': iris_df['sepal length (cm)'] * iris_df['sepal width (cm)'],
    'petal_area': iris_df['petal length (cm)'] * iris_df['petal width (cm)'],
    'target': iris_df['target']
})
sns.heatmap(
    iris_df_areas.phik_matrix(interval_cols=['sepal_area', 'petal_area']),
    annot=True,
    cmap='coolwarm'
)
plt.show()

Сразу становится заметна идеальная корреляция между площадью petal (лепестка) и видом цветка. Посмотрим еще раз на ящики с усами, но теперь по площади petal_area

In [ ]:
sns.boxplot(
    x=iris_df_areas['target'].map(dict(enumerate(iris.target_names))),
    y=iris_df_areas['petal_area'],
)
plt.show()

sns.boxplot(
    x=iris_df_areas['target'].map(dict(enumerate(iris.target_names))),
    y=iris_df_areas['sepal_area'],
)
plt.show()

# Обучение модели

In [ ]:
# Разделение данных на train и test.
# Отдельную validation-выборку не выделяем, так как для подбора гиперпараметров используется кросс-валидация (GridSearchCV) на train.
from sklearn.model_selection import train_test_split

X = iris_df.drop(columns=['target'])
y = iris_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

len(X_train), len(X_test)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
model = Pipeline([
    ('preprocessing', Pipeline([

    ])),
    ('model', LogisticRegression())
])